# MIDA507 -- Notebook Session 06
## Publication Open Data et Simulation d'API CKAN

| | |
|---|---|
| **Session** | S06 -- Publication open data et API CKAN |
| **Prerequis** | Notebooks S01-S05 |
| **Duree** | 70 a 90 minutes |

### Objectifs
1. Comprendre l'architecture CKAN (datasets, ressources, organisations, tags)
2. Simuler les appels API CKAN avec Python (sans serveur reel)
3. Construire un pipeline de publication complet : nettoyage -> DCAT -> publication
4. Verifier la publication et tester les endpoints API simules
5. Comprendre la souverainete des donnees africaines (hebergement local)

### Livrable : `MIDA507_S06_package_publication.json`


---
## PARTIE 1 -- Configuration


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns, json, os, random, hashlib
from datetime import datetime, timedelta, date
import warnings; warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
PAL={"ter1":"#3E1500","ter2":"#BF360C","ter3":"#E64A19",
     "olive":"#558B2F","bleu":"#1565C0","gris":"#546E7A","clair":"#FBE9E7"}
def regen(n=5000):
    random.seed(42); np.random.seed(42)
    g=["These et memoire","Manuel universitaire","Revue scientifique","Rapport de recherche","Ouvrage de reference","Document administratif"]
    fi=["Droit","Sciences eco.","Lettres modernes","Histoire","Geographie","Medecine","Agronomie","Informatique"]
    ni=["Licence 1","Licence 2","Licence 3","Master 1","Master 2","Doctorat"]
    re=["Adamaoua","Centre","Est","Extreme-Nord","Littoral","Nord","Ouest","Sud"]
    la=["Francais","Anglais","Bilingue FR/EN","Arabe","Autres lang. africaines"]
    li=["fra","eng","fra+eng","ara","mul"]
    d0=datetime(2018,1,1)
    df=pd.DataFrame({
        "ark_id":[f"ark:/67375/UNG-{str(i+1).zfill(6)}" for i in range(n)],
        "cote":[f"UNG-{str(i+1).zfill(5)}" for i in range(n)],
        "genre_document":np.random.choice(g,n,p=[0.28,0.30,0.15,0.10,0.10,0.07]),
        "date_emprunt":[(d0+timedelta(days=random.randint(0,365*5))).strftime("%Y-%m-%d") for _ in range(n)],
        "duree_jours":np.random.choice([3,7,14,21,30],n,p=[0.1,0.3,0.3,0.2,0.1]),
        "code_usager_anon":[f"USR-{hex(random.randint(0,0xFFFFFF))[2:].upper().zfill(6)}" for _ in range(n)],
        "filiere":np.random.choice(fi,n),
        "niveau":np.random.choice(ni,n),
        "region_origine":np.random.choice(re,n),
        "langue_document":np.random.choice(la,n,p=[0.62,0.22,0.10,0.04,0.02]),
        "langue_iso639":np.random.choice(li,n,p=[0.62,0.22,0.10,0.04,0.02]),
    })
    df["annee"]=pd.to_datetime(df["date_emprunt"]).dt.year
    return df
CSV="MIDA507_S05_bibliotheque_ung_nettoyee.csv"
if not os.path.exists(CSV): CSV="MIDA507_S02_bibliotheque_ung_enrichie.csv"
if os.path.exists(CSV):
    df=pd.read_csv(CSV); print(f"Jeu charge : {len(df):,} lignes")
else:
    df=regen(); print(f"Jeu regenere : {len(df):,} lignes")
print(f"Colonnes : {list(df.columns)}")


---
## PARTIE 2 -- Architecture CKAN

> CKAN (Comprehensive Knowledge Archive Network) est la plateforme open source
> de reference pour les portails open data. Elle est utilisee par data.gouv.bj,
> data.europa.eu, et des dizaines de portails africains.
> Notre objectif : simuler une publication CKAN complete avec Python.


In [ ]:
# Architecture CKAN : 5 entités principales
ARCHITECTURE_CKAN = {
    "Organization": {
        "description": "Institution proprietaire des jeux de donnees",
        "exemple": "Universite de Ngaoundere -- Bibliotheque Centrale",
        "champs_cles": ["name","title","description","image_url","is_organization"],
    },
    "Dataset (Package)": {
        "description": "Jeu de donnees complet avec ses metadonnees",
        "exemple": "ung-biblio-emprunts-2018-2023",
        "champs_cles": ["name","title","notes","tags","license_id",
                        "owner_org","maintainer","version","extras"],
    },
    "Resource": {
        "description": "Fichier ou API specifique associe a un dataset",
        "exemple": "ung_emprunts.csv (telechargement direct)",
        "champs_cles": ["name","url","format","mimetype","size","description"],
    },
    "Tag": {
        "description": "Mot-cle libre pour la recherche dans le catalogue",
        "exemple": "bibliotheque-universitaire, cameroun, emprunts",
        "champs_cles": ["name","display_name"],
    },
    "Group / Theme": {
        "description": "Categorie thematique (Education, Sante, Economie...)",
        "exemple": "education-et-formation",
        "champs_cles": ["name","title","type"],
    }
}

fig, ax = plt.subplots(figsize=(13,7))
ax.set_xlim(0,14); ax.set_ylim(0,9); ax.axis("off")
couleurs = ["#3E1500","#BF360C","#1565C0","#558B2F","#546E7A"]
positions = [(1,4.5),(4.5,7.5),(8,4.5),(4.5,1.5),(11,7.5)]
for (entite, info), (x,y), col in zip(ARCHITECTURE_CKAN.items(), positions, couleurs):
    ax.add_patch(plt.FancyBboxPatch((x-0.9,y-0.7),3.5,1.4,
                 boxstyle="round,pad=0.1",facecolor=col,edgecolor="white",linewidth=2))
    ax.text(x+0.85,y+0.2,entite,ha="center",va="center",fontsize=9,fontweight="bold",color="white")
    ax.text(x+0.85,y-0.25,info["exemple"][:35],ha="center",va="center",fontsize=7.5,color="#DDD")

# Fleches de relations
for (x1,y1),(x2,y2) in [((1,4.5),(4.5,7.5)),((4.5,7.5),(8,4.5)),
                          ((4.5,7.5),(4.5,1.5)),((4.5,7.5),(11,7.5))]:
    ax.annotate("",xy=(x2-0.9,y2),xytext=(x1+2.6,y1),
                arrowprops=dict(arrowstyle="->",color="#888",lw=1.5))

ax.text(7,0.5,"contient",ha="center",fontsize=8,color="#555",style="italic")
ax.set_title("Architecture CKAN -- 5 entites et leurs relations",
             fontsize=13,fontweight="bold",color="#3E1500")
plt.tight_layout(); plt.show()
print("Prochaine etape : simuler la publication de notre jeu BU-UNG dans CKAN.")


---
## PARTIE 3 -- Simulation de l'API CKAN

> L'API CKAN REST permet de creer, lire, modifier et supprimer des jeux de donnees
> via des requetes HTTP standard. On la simule ici en Python.


In [ ]:
class CKANSimulator:
    """Simule une instance CKAN avec API REST (sans serveur reel)."""

    BASE_URL = "https://ckan.demo.ung.cm"

    def __init__(self):
        self._organisations = {}
        self._datasets = {}
        self._resources = {}
        self._tags = {}
        self._log_appels_api = []

    def _log(self, endpoint, methode, statut, details=""):
        self._log_appels_api.append({
            "timestamp": datetime.now().strftime("%H:%M:%S"),
            "endpoint": endpoint,
            "methode": methode,
            "statut": statut,
            "details": details
        })

    def organization_create(self, name, title, description, image_url=None):
        """POST /api/3/action/organization_create"""
        if name in self._organisations:
            self._log("/api/3/action/organization_create","POST","409 Conflict",f"Organisation '{name}' deja existante")
            return {"success":False,"error":{"message":f"Organisation '{name}' already exists"}}
        org = {"id":f"org-{len(self._organisations)+1}","name":name,
               "title":title,"description":description,"image_url":image_url,
               "package_count":0,"is_organization":True}
        self._organisations[name] = org
        self._log("/api/3/action/organization_create","POST","200 OK",f"Organisation '{name}' creee")
        return {"success":True,"result":org}

    def package_create(self, name, title, notes, owner_org, license_id,
                        tags, maintainer, version, extras=None):
        """POST /api/3/action/package_create"""
        if owner_org not in self._organisations:
            self._log("/api/3/action/package_create","POST","404 Not Found",f"Organisation '{owner_org}' inexistante")
            return {"success":False,"error":{"message":f"Organisation '{owner_org}' not found"}}
        if name in self._datasets:
            self._log("/api/3/action/package_create","POST","409 Conflict",f"Dataset '{name}' deja existant")
            return {"success":False,"error":{"message":f"Dataset '{name}' already exists"}}

        pkg = {"id":f"pkg-{len(self._datasets)+1}","name":name,"title":title,
               "notes":notes,"owner_org":owner_org,"license_id":license_id,
               "tags":[{"name":t} for t in tags],"maintainer":maintainer,
               "version":version,"num_resources":0,"state":"active",
               "extras":extras or [],"url":f"{self.BASE_URL}/dataset/{name}",
               "created":datetime.now().isoformat()}
        self._datasets[name] = pkg
        self._organisations[owner_org]["package_count"] += 1
        self._log("/api/3/action/package_create","POST","200 OK",f"Dataset '{name}' cree")
        return {"success":True,"result":pkg}

    def resource_create(self, package_id, name, url, format_, mimetype, size, description=""):
        """POST /api/3/action/resource_create"""
        pkg = next((p for p in self._datasets.values() if p["id"]==package_id or p["name"]==package_id), None)
        if not pkg:
            self._log("/api/3/action/resource_create","POST","404 Not Found","Dataset introuvable")
            return {"success":False,"error":{"message":"Package not found"}}
        res = {"id":f"res-{len(self._resources)+1}","package_id":pkg["id"],
               "name":name,"url":url,"format":format_,"mimetype":mimetype,
               "size":size,"description":description,"created":datetime.now().isoformat()}
        self._resources[res["id"]] = res
        pkg["num_resources"] += 1
        self._log("/api/3/action/resource_create","POST","200 OK",f"Resource '{name}' ajoutee")
        return {"success":True,"result":res}

    def package_show(self, name):
        """GET /api/3/action/package_show?id=NAME"""
        if name in self._datasets:
            self._log(f"/api/3/action/package_show?id={name}","GET","200 OK","")
            return {"success":True,"result":self._datasets[name]}
        self._log(f"/api/3/action/package_show?id={name}","GET","404 Not Found","")
        return {"success":False,"error":{"message":"Not found"}}

    def datastore_search(self, resource_id, filters=None, limit=10, offset=0):
        """GET /api/3/action/datastore_search (simule avec le DataFrame)"""
        self._log(f"/api/3/action/datastore_search?resource_id={resource_id}","GET","200 OK",
                  f"filters={filters}, limit={limit}")
        return {"success":True,"result":{"total":len(df),"records":"[simulation] voir le DataFrame",
                "fields":[{"id":c,"type":"text"} for c in df.columns][:5]}}

    def afficher_log(self, n=10):
        print(f"JOURNAL API CKAN (derniers {min(n,len(self._log_appels_api))} appels)")
        print("-"*65)
        for appel in self._log_appels_api[-n:]:
            print(f"  [{appel['timestamp']}] {appel['methode']:<6} {appel['statut']:<20} {appel['endpoint']}")

ckan = CKANSimulator()
print("Instance CKAN simulee initialisee.")
print(f"URL de base : {ckan.BASE_URL}")


---
## PARTIE 4 -- Pipeline de Publication Complet


In [ ]:
print("PIPELINE DE PUBLICATION BU-UNG --> CKAN")
print("="*60)

# ETAPE 1 : Creer l'organisation
print("\n[1/4] Creation de l'organisation...")
r1 = ckan.organization_create(
    name        = "univ-ngaoundere-bibliotheque",
    title       = "Universite de Ngaoundere -- Bibliotheque Centrale",
    description = "Bibliotheque universitaire centrale desservant 12 000 etudiants. Cameroun.",
    image_url   = "https://www.univ-ngaoundere.cm/logo.png"
)
print(f"  Statut : {'OK' if r1['success'] else 'ERREUR'}")

# ETAPE 2 : Creer le dataset
print("\n[2/4] Creation du dataset...")
r2 = ckan.package_create(
    name       = "ung-biblio-emprunts-2018-2023",
    title      = "Emprunts documentaires -- Bibliotheque UNG 2018-2023",
    notes      = f"Enregistrements d'emprunt de la BU-UNG. {len(df):,} transactions. Periode 2018-2023.",
    owner_org  = "univ-ngaoundere-bibliotheque",
    license_id = "cc-by",
    tags       = ["bibliotheque-universitaire","cameroun","emprunts","education","mida507"],
    maintainer = "Data Steward IDA -- MIDA507",
    version    = "1.0",
    extras     = [{"key":"dcat:identifier","value":"ark:/67375/UNG-BIBLIO-2018-2023"},
                  {"key":"score_fair","value":"87"},
                  {"key":"standard_metadonnees","value":"DCAT-AP"}]
)
pkg_name = r2.get("result",{}).get("name","ung-biblio-emprunts-2018-2023")
print(f"  Statut : {'OK' if r2['success'] else 'ERREUR'}")
if r2["success"]:
    print(f"  URL    : {r2['result']['url']}")

# ETAPE 3 : Ajouter les ressources
print("\n[3/4] Ajout des ressources...")
r3a = ckan.resource_create(
    package_id  = pkg_name,
    name        = "ung_emprunts_2018_2023.csv",
    url         = f"{ckan.BASE_URL}/dataset/ung-biblio/ung_emprunts_2018_2023.csv",
    format_     = "CSV",
    mimetype    = "text/csv; charset=UTF-8",
    size        = len(df)*len(df.columns)*15,
    description = f"Fichier plat CSV UTF-8 avec BOM. {len(df):,} lignes, {len(df.columns)} colonnes."
)
print(f"  CSV  : {'OK' if r3a['success'] else 'ERREUR'}")

r3b = ckan.resource_create(
    package_id  = pkg_name,
    name        = "API JSON -- CKAN Datastore",
    url         = f"{ckan.BASE_URL}/api/3/action/datastore_search?resource_id=ung-emprunts",
    format_     = "API",
    mimetype    = "application/json",
    size        = 0,
    description = "API REST JSON avec filtres par genre, region, annee et langue."
)
print(f"  API  : {'OK' if r3b['success'] else 'ERREUR'}")

# ETAPE 4 : Verification
print("\n[4/4] Verification de la publication...")
r4 = ckan.package_show(pkg_name)
if r4["success"]:
    pkg = r4["result"]
    print(f"  Dataset : {pkg['title']}")
    print(f"  Org     : {pkg['owner_org']}")
    print(f"  Licence : {pkg['license_id']}")
    print(f"  Ressources : {pkg['num_resources']}")
    print(f"  Tags : {', '.join(t['name'] for t in pkg['tags'])}")

print()
ckan.afficher_log()


In [ ]:
# Tests des endpoints API CKAN simulés
print("\nTESTS API CKAN -- Requetes des usagers")
print("="*60)

tests = [
    ("Chercheur universitaire -- telecharger les donnees brutes",
     "/api/3/action/package_show?id=ung-biblio-emprunts-2018-2023"),
    ("ONG agricole -- filtrer par region Adamaoua",
     "/api/3/action/datastore_search?resource_id=ung-emprunts&filters={"region":"Adamaoua"}&limit=100"),
    ("Journaliste -- agregats par annee",
     "/api/3/action/datastore_search?resource_id=ung-emprunts&fields=annee,genre_document&limit=10"),
]

for usager, endpoint in tests:
    print(f"\n  Usager   : {usager}")
    print(f"  Endpoint : {endpoint[:70]}...")
    # Simuler la reponse
    if "package_show" in endpoint:
        res = ckan.package_show(pkg_name)
    else:
        parts = endpoint.split("?",1)
        res = ckan.datastore_search("ung-emprunts", limit=10)
    print(f"  Statut   : {'200 OK' if res.get('success') else '404 Not Found'}")
    if res.get("success") and "result" in res:
        r = res["result"]
        if "total" in r:
            print(f"  Total enregistrements : {r['total']:,}")
        elif "title" in r:
            print(f"  Dataset : {r['title']}")

# Sauvegarde du package de publication
package_pub = {
    "ckan_instance": ckan.BASE_URL,
    "date_publication": datetime.now().isoformat(),
    "organisation": ckan._organisations.get("univ-ngaoundere-bibliotheque",{}),
    "dataset": ckan._datasets.get(pkg_name,{}),
    "ressources": list(ckan._resources.values()),
    "log_api": ckan._log_appels_api,
    "score_fair_cible": 87
}
with open("MIDA507_S06_package_publication.json","w",encoding="utf-8") as f:
    json.dump(package_pub,f,ensure_ascii=False,indent=2)
print("\nPackage de publication sauvegarde : MIDA507_S06_package_publication.json")


---
### EXERCICE -- Planifiez la publication de votre jeu de donnees


In [ ]:
# ============================================================
# EXERCICE -- Planification de publication CKAN
# ============================================================
# Completez les informations ci-dessous pour planifier
# la publication d'un jeu de donnees de votre institution.

MON_ORG_NAME  = "[nom-institution-sans-espaces]"     # <- ex: ins-cameroun
MON_ORG_TITRE = "[Nom complet de l'institution]"     # <- ex: Institut National de la Statistique

MON_DATASET_NAME  = "[nom-dataset-sans-espaces]"     # <- ex: enquete-emploi-2023
MON_DATASET_TITRE = "[Titre complet et descriptif]"  # <- titre long et descriptif
MES_TAGS          = ["[tag1]","[tag2]","[tag3]"]     # <- au moins 5 tags
MA_LICENCE        = "cc-by"                          # <- recommande : cc-by

MON_CSV_URL = f"https://ckan.demo.{MON_ORG_NAME}.cm/dataset/{MON_DATASET_NAME}/data.csv"

# Verification
champs_remplis = [
    ("Organisation",    MON_ORG_NAME,    "[" not in MON_ORG_NAME),
    ("Titre org",       MON_ORG_TITRE,   "[" not in MON_ORG_TITRE),
    ("Nom dataset",     MON_DATASET_NAME,"[" not in MON_DATASET_NAME),
    ("Titre dataset",   MON_DATASET_TITRE,"[" not in MON_DATASET_TITRE),
    ("Tags (>=3)",      str(MES_TAGS),   "[" not in str(MES_TAGS) and len(MES_TAGS)>=3),
    ("URL CSV",         MON_CSV_URL,     "[" not in MON_ORG_NAME),
]
print("PLAN DE PUBLICATION -- MON INSTITUTION")
print("="*55)
tout_ok = True
for champ, val, ok in champs_remplis:
    print(f"  {'OK' if ok else 'A REMPLIR'} {champ:<20} : {str(val)[:45]}")
    if not ok: tout_ok = False

if tout_ok:
    ckan2 = CKANSimulator()
    ckan2.organization_create(name=MON_ORG_NAME,title=MON_ORG_TITRE,description="")
    r = ckan2.package_create(name=MON_DATASET_NAME,title=MON_DATASET_TITRE,
                              notes="",owner_org=MON_ORG_NAME,license_id=MA_LICENCE,
                              tags=MES_TAGS,maintainer="Data Steward IDA",version="1.0")
    if r["success"]:
        print(f"\nSimulation publication reussie !")
        print(f"URL : {r['result']['url']}")
else:
    print("\n[A COMPLETER] Remplissez les champs entre crochets ci-dessus.")


---
## Bilan S06

| Competence | Statut |
|---|---|
| Comprendre l'architecture CKAN | OK |
| Appeler l'API CKAN avec Python | OK |
| Executer un pipeline de publication complet | OK |
| Tester les endpoints pour les 3 types d'usagers | OK |
| Planifier la publication de mon propre jeu | A completer |

**Lien S07 :** Avant toute publication reelle, les codes usagers seront anonymises (k-anonymite).

*Notebook MIDA507 S06 -- Master MIDA -- Universite de Douala*
